In [ ]:
import kan, torch
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from kan import ex_round

def normalize(x):  # min-max scaling
    if x.min() == x.max():
        return (x*0 + 1)
    return (x - x.min()) / (x.max() - x.min())

def denormalize(x, x_min, x_max):  # min-max unscaling
    return x * (x_max - x_min) + x_min

def train_model(expmnt, kan_width, kan_grid, kan_k, all_time, all_PCD, all_volt, all_bias, all_load, 
                time_mx_mn, PCD_mx_mn, volt_mx_mn, train_losses, test_losses, train_ratio):
    # kan_width must be a list
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    #print(device)

    kan_inputs = 4 # time, PCD, bias, load
    # initialize KAN
    model = kan.KAN(width=kan_width, grid=kan_grid, k=kan_k, seed=1, device=str(device))

    # declare variables used elsewhere in program.
    expmnt_size = []

    for exp_num in expmnt:
        if exp_num == 0: continue
        # collect data from dataset
        file_name = f"TestData\\LFXR_Lovejoy_Sept2021_{exp_num}.csv"
        df_train = pd.read_csv(file_name, usecols=[0,1,3])

        df_start_stop = pd.read_csv("start_stop_pts.csv")
        expmnt_row = exp_num - 27271   # the first experiment is on row 1
        volt_start = int(df_start_stop['volt_start'].iloc[expmnt_row])
        volt_stop = int(df_start_stop['volt_stop'].iloc[expmnt_row])
        rad_start = int(df_start_stop['rad_start'].iloc[expmnt_row])
        rad_stop = int(df_start_stop['rad_stop'].iloc[expmnt_row])

        expmnt_size.append(volt_stop - volt_start)

        bias = float(df_start_stop['bias'].iloc[expmnt_row])
        load = float(df_start_stop['load'].iloc[expmnt_row])

        
        #print(volt_start, volt_stop, rad_start, rad_stop)

        time = df_train['time1'].iloc[volt_start:volt_stop].to_numpy(dtype = np.float32)
        # Make time in microseconds
        time = time * 1e6
        time = time - time.min()  # shift time to start at 0
        time_mx_mn[0].append(time.max())
        time_mx_mn[1].append(time.min())    
        time = normalize(time)  # normalize time to [0,1]

        PCD = df_train['PCD3_B'].iloc[rad_start:rad_stop].to_numpy(dtype = np.float32)
        PCD_mx_mn[0].append(PCD.max())
        PCD_mx_mn[1].append(PCD.min())  
        PCD = normalize(PCD)  # normalize PCD to [0,1]

        # output voltage
        volt = df_train['Diode'].iloc[volt_start:volt_stop].to_numpy(dtype = np.float32).reshape(-1,1)
        volt = volt / (bias if bias > 1 else 1)
        volt_mx_mn[0].append(volt.max())
        volt_mx_mn[1].append(volt.min())
        volt = normalize(volt)  # normalize voltage to [0,1]

        all_time.append(time)
        all_PCD.append(PCD)
        all_volt.append(volt)
        all_bias.append(np.full_like(time, bias))
        all_load.append(np.full_like(time, np.log10(load))) # use logarithmic scale for load to prevent NANs.
        

    all_time = np.concatenate(all_time)
    all_PCD = np.concatenate(all_PCD)
    all_volt = np.concatenate(all_volt)
    all_bias = np.concatenate(all_bias)
    all_load = np.concatenate(all_load)

    # Normalize the data
    n_time = all_time  # normalize time to [0,1]
    n_PCD = all_PCD  # normalize PCD to [0,1]
    # normalize voltage to [0,1]
    n_volt = all_volt  # normalize voltage to [0,1] but that is already happening in the for loop

    n_bias = all_bias
    n_load = all_load
    # Diode will be one-hot encoded 

    # Make input matrix
    X = np.column_stack((n_time, n_PCD, n_bias, n_load))  # shape (num_samples, 8)


    #pull training points from the data
    np.random.seed(None)
    # need to get certain number of points from each experiment, so we don't just pull the first 70% of the data
    slice_idx = 0
    test_indices = np.array([], dtype=int)
    
    for idx, i in enumerate(expmnt):
        if i == 0: continue
        slice_idx += expmnt_size[idx]
        exp_indices = np.arange(slice_idx - expmnt_size[idx], slice_idx)
        exp_test_indices = np.random.choice(exp_indices, 
                            round(expmnt_size[idx] * (1 - train_ratio)), replace=False)
        test_indices = np.concatenate((test_indices, exp_test_indices))

    X_test = X[test_indices]
    n_volt_test = n_volt[test_indices] # Correct Answers

    #pull the rest for training
    train_indices = np.setdiff1d(np.arange(X.shape[0]), test_indices)
    X_train = X[train_indices]
    n_volt_train = n_volt[train_indices]

    dataset = {
        "train_input": torch.from_numpy(X_train).to(device),
        "train_label": torch.from_numpy(n_volt_train).to(device),
        "test_input":  torch.from_numpy(X_test).to(device),
        "test_label":  torch.from_numpy(n_volt_test).to(device),
    }

    print(X.shape, n_volt.shape)
    

    results = model.fit(dataset,
            opt="Adam", # Adam (faster) or LBFGS (slower)
            steps=500,
            lr=0.016,
            update_grid=True,  # set to false for single experiment training.
            start_grid_update_step= 60,
            lamb=0.6,        # master scale — tune this first
            lamb_l1=0.0,      # high — you want to prune weak connections
            lamb_entropy=0.0, # high — encourages interpretable simple functions
            lamb_coef=0.0,    # usually redundant if l1 is active
            lamb_coefdiff=0.0003) # smooth splines — important for noisy data)
    train_losses += results['train_loss']
    test_losses += results['test_loss']

    # # plot the two losses
    # plt.plot(train_losses)
    # plt.plot(test_losses)
    # plt.legend(['train', 'test'])
    # plt.ylabel('RMSE')
    # plt.xlabel('step')
    # plt.yscale('log')
    # plt.show()

    return model, dataset, expmnt_size





In [26]:
def plot_results(model, dataset, time_mx_mn, PCD_mx_mn, volt_mx_mn, expmnt, expmnt_size, train_ratio):    

    t = dataset["test_input"][:, 0]   # assumes column 0 is time
    r = dataset["test_input"][:, 1]   # assumes column 1 is radiation
    v = dataset["test_label"]         # voltage target
    b = dataset["test_input"][:, 2]   # assumse column 2 is radiation

    with torch.no_grad():
        v_pred = model(dataset["test_input"])
    # MSE and RMSE in percentage form
    mse = torch.mean((v_pred - v) ** 2).item() 
    print(f"{expmnt} Test MSE: {mse * 100:.3f}%")
    rmse = mse ** 0.5
    print(f"{expmnt} Test RMSE: {rmse * 100:.3f}%")

    test_results_idx = 0
    # test_indice are already defined
    for idx, i in enumerate(expmnt):
        if i == 0: continue
        # Get the number of test points for this experiment
        num_test_pts = round(expmnt_size[idx] * (1 - train_ratio))

        # Extract the relevant portion of the data for this experiment
        t_np = t.detach().cpu().numpy()[test_results_idx:test_results_idx + num_test_pts]
        r_np = r.detach().cpu().numpy()[test_results_idx:test_results_idx + num_test_pts]
        v_np = v.detach().cpu().numpy()[test_results_idx:test_results_idx + num_test_pts]
        v_pred_np = v_pred.detach().cpu().numpy()[test_results_idx:test_results_idx + num_test_pts]
        bias = b[test_results_idx]

        # Denormalize the data for plotting ; xxxx_mx_min[0] is max, xxxx_mx_mn[1] is min
        t_np = denormalize(t_np, time_mx_mn[1][idx], time_mx_mn[0][idx])  # unnormalize time
        r_np = denormalize(r_np, PCD_mx_mn[1][idx], PCD_mx_mn[0][idx])  # unnormalize PCD
        v_np = denormalize(v_np, volt_mx_mn[1][idx], volt_mx_mn[0][idx])  * float(bias if bias > 1 else 1) # unnormalize voltage
        v_pred_np = denormalize(v_pred_np, volt_mx_mn[1][idx], volt_mx_mn[0][idx]) * float(bias if bias > 1 else 1) # unnormalize voltage

        test_results_idx += num_test_pts

        arr = np.column_stack((t_np / 1e6, r_np))
        sorted_arr = arr[np.argsort(arr[:, 0])]
        # Define folder and file path
        folder = "pulse_text_files"
        os.makedirs(folder, exist_ok=True)  # Creates folder if it doesn't exist

        file_path = os.path.join(folder, f"pulse_{i}.txt")

        # # Save array to .txt file
        # np.savetxt(
        #     file_path,
        #     sorted_arr,
        #     fmt="%.9e",          # Integer format (use "%.4f" for floats)
        #     delimiter=" ",     # Whitespace delimiter
        #     )

        plt.figure()
        plt.scatter(t_np, v_np, s=20, label="voltage (true)")
        plt.scatter(t_np, v_pred_np, s=7, label="voltage (pred)")
        #plt.scatter(t_np, r_np, s=7, label="radiation")
        plt.xlabel("time (us)")
        plt.ylabel("voltage (V)")
        plt.title(f"Experiment {i}")
        plt.legend()
        plt.show()

# # Set nodes with functions
# kan.add_symbolic('exp_neg', lambda x: torch.exp(-x))
# kan.add_symbolic('sigmoid', lambda x: torch.sigmoid(x))
# kan.add_symbolic('gaussian', lambda x: torch.exp(-x**2))

# lib = [
#     'x',
#     'exp',
#     'exp_neg',
#     'log',
#     'sqrt',
#     'tanh',
#     'sigmoid',
#     'gaussian',
#     'sin',
#     'abs',
# ]

# model = model.prune()
# model.auto_symbolic(lib=lib, r2_threshold=0.88)
# formula = model.symbolic_formula()[0][0]
# kan.ex_round(formula, 4)


# Used to test the model using a single experiment to test generalization.

def single_test(model, exp_num):
    print(f"Testing with experiment {exp_num}...")
    #reset variables
    all_time = []
    time_mx_mn = [[],[]]
    all_PCD = []
    PCD_mx_mn = [[],[]]
    all_volt = []
    volt_mx_mn = [[],[]]
    all_bias = []
    all_load = []


    expmnt_size = []
    
    # collect data from dataset
    file_name = f"TestData\\LFXR_Lovejoy_Sept2021_{exp_num}.csv"
    df_train = pd.read_csv(file_name, usecols=[0,1,3])

    df_start_stop = pd.read_csv("start_stop_pts.csv")
    expmnt_row = exp_num - 27271   # the first experiment is on row 1
    volt_start = int(df_start_stop['volt_start'].iloc[expmnt_row])
    volt_stop = int(df_start_stop['volt_stop'].iloc[expmnt_row])
    rad_start = int(df_start_stop['rad_start'].iloc[expmnt_row])
    rad_stop = int(df_start_stop['rad_stop'].iloc[expmnt_row])

    expmnt_size.append(volt_stop - volt_start)

    bias = float(df_start_stop['bias'].iloc[expmnt_row])
    load = float(df_start_stop['load'].iloc[expmnt_row])


    
    #print(volt_start, volt_stop, rad_start, rad_stop)

    time = df_train['time1'].iloc[volt_start:volt_stop].to_numpy(dtype = np.float32)
    # Make time in microseconds
    time = time * 1e6
    time = time - time.min()  # shift time to start at 0
    time_mx_mn[0].append(time.max())
    time_mx_mn[1].append(time.min())    
    time = normalize(time)  # normalize time to [0,1]

    PCD = df_train['PCD3_B'].iloc[rad_start:rad_stop].to_numpy(dtype = np.float32)
    PCD_mx_mn[0].append(PCD.max())
    PCD_mx_mn[1].append(PCD.min())  
    PCD = normalize(PCD)  # normalize PCD to [0,1]

    # output voltage
    volt = df_train['Diode'].iloc[volt_start:volt_stop].to_numpy(dtype = np.float32).reshape(-1,1)
    volt = volt / (bias if bias > 1 else 1)
    volt_mx_mn[0].append(volt.max())
    volt_mx_mn[1].append(volt.min())
    volt = normalize(volt)  # normalize voltage to [0,1]

    all_time.append(time)
    all_PCD.append(PCD)
    all_volt.append(volt)
    all_bias.append(np.full_like(time, bias))
    all_load.append(np.full_like(time, np.log10(load))) # use logarithmic scale for load to prevent NANs.


    all_time = np.concatenate(all_time)
    all_PCD = np.concatenate(all_PCD)
    all_volt = np.concatenate(all_volt)
    all_bias = np.concatenate(all_bias)
    all_load = np.concatenate(all_load)

    # Normalize the data
    n_time = normalize(all_time)  # normalize time to [0,1]
    n_PCD = normalize(all_PCD)  # normalize PCD to [0,1]
    # normalize voltage to [0,1]
    n_volt = normalize(all_volt)  # normalize voltage to [0,1]

    n_bias = all_bias
    n_load = all_load
    # Diode will be one-hot encoded 

    # Make input matrix
    X = np.column_stack((n_time, n_PCD, n_bias, n_load))  # shape (num_samples, 8)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = {
        #"train_input": torch.from_numpy(X).to(device),
        #"train_label": torch.from_numpy(n_volt).to(device),
        "test_input":  torch.from_numpy(X).to(device),
        "test_label":  torch.from_numpy(n_volt).to(device),
    }
    exp_num = [exp_num]
    plot_results(model, dataset, time_mx_mn, PCD_mx_mn, volt_mx_mn, exp_num, expmnt_size, train_ratio = 0)

    

In [ ]:
def perform_training(kan_width, kan_grid, kan_k, LOO = False):

    # IF something does not work, the bias is being used to scale down voltage and then scale it up again in plotting.
    if LOO:
        filename = "LOO_training_subsets.csv"
        subsets_df = pd.read_csv(filename)
        train_these = subsets_df.drop(columns=["Name", "Test", "Reasoning"]).to_numpy().tolist()
    else:
        filename = "training_subsets.csv"
        subsets_df = pd.read_csv(filename)
        train_these = subsets_df.drop(columns=["Name", "Reasoning"]).to_numpy().tolist()


    final_train_losses = []
    for i in train_these:
        print(f"Processing experiment {i}...")
        #reset variables
        all_time = []
        time_mx_mn = [[],[]]
        all_PCD = []
        PCD_mx_mn = [[],[]]
        all_volt = []
        volt_mx_mn = [[],[]]
        all_bias = []
        all_load = []

        train_losses = []
        test_losses = []
        train_ratio = 0.7  # Use 70% of the data for training
        if any(item in i for item in [27277, 27292, 27293, 27306]): 
            print(f"This subset {i} has an unusable experiment. Skipping")
        else:
            expmnt = i


            M1, data1, expmnt_size = train_model(expmnt, kan_width, kan_grid, kan_k, all_time, all_PCD, all_volt, all_bias, 
                                                all_load, time_mx_mn, PCD_mx_mn, 
                                                volt_mx_mn, train_losses, test_losses, train_ratio)
            final_train_losses.append(round(float(test_losses[-1])*100,3)) # already in percentage form and rounded
            #plot_results(M1, data1, time_mx_mn, PCD_mx_mn, volt_mx_mn, expmnt, expmnt_size, train_ratio)
    train_loss_df = pd.DataFrame({"Name"  : subsets_df["Name"], 
                                    "Error" : final_train_losses,
                                    "Reasoning": subsets_df["Reasoning"]})
    train_loss_df.to_csv("final_train_loss.csv", index=False)
    print(f"Train Error Rates: {final_train_losses}")




perform_training([4,10,1], 10, 3)

Processing experiment [27296, 27297, 27298, 0, 0]...
checkpoint directory created: ./model
saving model version 0.0
(1800, 4) (1800, 1)


| train_loss: 7.23e-02 | test_loss: 7.45e-02 | reg: 2.62e-03 | : 100%|█| 500/500 [00:11<00:00, 43.00


saving model version 0.1
Processing experiment [27281, 27282, 27283, 27284, 0]...
checkpoint directory created: ./model
saving model version 0.0
(2400, 4) (2400, 1)


| train_loss: 1.35e-01 | test_loss: 1.45e-01 | reg: 2.22e-03 | : 100%|█| 500/500 [00:12<00:00, 41.34

saving model version 0.1


In [ ]:
def perform_LOO_training():
    subsets_df = pd.read_csv("training_subsets.csv")
    LOO_subsets_df = pd.DataFrame(columns=["Name", "Exp01", "Exp02", "Exp03", "Exp04", "Test", "Reasoning"])

    for i in range(subsets_df.shape[0]):
        row = subsets_df.iloc[i,:].to_list()
        if row[3] == 0: continue # make sure experiment has at least 3 experiments
        for j in row[1:-1]:
            train = row[1:-1].pop(j)
            new_row = [row[0]]
            new_row += train
            new_row += [j, row[-1]]
            LOO_subsets_df.loc[len(LOO_subsets_df)] = new_row

    LOO_subsets_df.to_csv("LOO_training_subsets.csv", index = False)
        
